# 04 · ESM1b — a protein language model

ESM1b (Brandes et al. 2023, *Nat Genet*, PMID 37563329) is an *LLM for protein sequences*. Like EVE it is **unsupervised**, but it scores variants on a **log-likelihood ratio (LLR)** whose scale runs *backwards* from every other tool here.

> ### The 3 CFTR UniProt IDs (P13569, P13569-2, P13569-3)
> ESM1b variant files (ntranoslab) list **three** CFTR-related isoforms:
> **P13569** is the **canonical** CFTR isoform (1480 aa; matches MANE `NM_000492.4`),
> while **-2** and **-3** are alternative UniProt isoforms (differ by alternative
> splicing). **Use the canonical `P13569`** so residue numbering lines up with
> AlphaMissense / CFTR2 / gnomAD — picking `-2`/`-3` silently shifts positions and
> breaks the `protein_variant` join. *(Isoform details: UniProt P13569.)*

> ✅ **REAL DATA.** Full CFTR **saturation** ESM1b LLR — **~28,120** variants (all 1,480 residues), `data/esm1b_cftr.csv`, built by the cell below from a manually-downloaded release zip (ntranoslab, canonical UniProt **P13569**). `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · ESM1b — a protein language model

**What is it?** ESM1b is a **protein language model** — think of it as an *LLM for protein sequences*. Just as a text LLM learns which word is likely to come next, ESM1b was trained on millions of natural protein sequences to predict which amino acid "belongs" at each position given its neighbours. It never saw the alignment explicitly; it learned protein grammar directly from raw sequences.

To score a variant, ESM1b compares how *likely* the model thinks the **mutant** amino acid is versus the **wild-type** amino acid at that position. This is a **log-likelihood ratio (LLR)**:

$$\text{LLR} = \log \frac{P(\text{mutant amino acid})}{P(\text{wild-type amino acid})}$$

> ### 🔄 IMPORTANT — ESM1b runs *backwards* from the other tools
> A **more negative** LLR means the model finds the mutation more *surprising* → **more damaging**. This is the **opposite direction** to EVE, AlphaMissense, and REVEL, where *higher* = worse.
> 
> - **Cut-off:** `LLR <= -7.5` ~ pathogenic. **Lower (more negative) = worse.**
> - Unsupervised (learned from sequences only) → low circularity, just like EVE.

## Building the REAL data — a manual download (bulk release, no per-gene API)

ESM1b has no API — the ntranoslab team publishes per-isoform LLR matrices for
**every human protein** (~42,000 files) as one zip. **You must fetch this one
yourself:**

1. Go to the HuggingFace Space **`ntranoslab/esm_variants`**
   (github.com/ntranoslab/esm-variants) and download
   **`ALL_hum_isoforms_ESM1b_LLR.zip`**.
2. Save it as `data/ALL_hum_isoforms_ESM1b_LLR.zip` (gitignored — never commit it).

The cell below opens the zip **without extracting it**, reads only CFTR's one
member (`P13569_LLR.csv` — **P13569 is the canonical isoform**, see the callout
above), melts its wide LLR matrix (columns = wild-type+position, rows = mutant
amino acid) into the long `protein_variant` form every other tool uses, and
writes `data/esm1b_cftr.csv`. Unlike EVE, this is **full saturation**: all 1,480
residues × 19 substitutions (~28,120 rows) — the matrix has no missing cells.

License: code is MIT; the scores themselves are released "per publication"
(Brandes et al. 2023) rather than under an explicit redistribution license —
see `data_manifest.json`.

**Version & reproducibility.** Same situation as EVE (tools/03): this is a
manual, authenticated download, not something this cell fetches over HTTP, so
there's no `Last-Modified` header to grab. But the zip is still versioned —
every member file inside a zip archive carries the timestamp of when it was
packaged, readable straight from the archive's own metadata
(`zipfile.ZipInfo.date_time`), no download needed. For
`content/ALL_hum_isoforms_ESM1b_LLR/P13569_LLR.csv`, that embedded timestamp is
**2022-05-06 05:28:04** (neighboring isoform files in the same zip carry
timestamps within minutes of it, consistent with one batch export). The build
cell records it into `data/esm1b_cftr.release.json`; `load_esm1b()` exposes it
as the `esm1b_release` column, the same way `eve_release` works for EVE and
`am_release` works for AlphaMissense.


In [2]:
import zipfile, io, json
from datetime import datetime

DATA_DIR = pathlib.Path.cwd().parent / "data"
ESM1B_ZIP = DATA_DIR / "ALL_hum_isoforms_ESM1b_LLR.zip"
ESM1B_MEMBER = "content/ALL_hum_isoforms_ESM1b_LLR/P13569_LLR.csv"
ESM1B_TSV = DATA_DIR / "esm1b_cftr.csv"
ESM1B_RELEASE_JSON = DATA_DIR / "esm1b_cftr.release.json"

if ESM1B_TSV.exists():
    print(f"already built -> {ESM1B_TSV.name} (delete it and {ESM1B_RELEASE_JSON.name} to rebuild)")
elif not ESM1B_ZIP.exists():
    raise FileNotFoundError(
        f"{ESM1B_ZIP} not found.\n"
        "ESM1b has no per-gene API -- get the bulk release (one <UniProt>_LLR.csv per\n"
        "human isoform, ~42,000 files, packaged as one zip):\n"
        "  1. Go to the HuggingFace Space 'ntranoslab/esm_variants'\n"
        "     (github.com/ntranoslab/esm-variants) and download\n"
        "     'ALL_hum_isoforms_ESM1b_LLR.zip'\n"
        f"  2. Save it as {ESM1B_ZIP} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it reads only CFTR's one member file (P13569_LLR.csv)\n"
        "from inside the zip, never extracting the other ~42,000."
    )
else:
    with zipfile.ZipFile(ESM1B_ZIP) as z:
        member_info = z.getinfo(ESM1B_MEMBER)
        # The zip's own internal metadata: when ntranoslab packaged this exact
        # file, straight from the archive -- no download or HTTP call needed.
        packaged_at = datetime(*member_info.date_time).isoformat()
        with z.open(ESM1B_MEMBER) as fh:
            m = pd.read_csv(io.TextIOWrapper(fh, encoding="utf-8"), index_col=0)
    print(f"P13569_LLR.csv packaged at (zip-embedded timestamp): {packaged_at}")
    # m is an LLR matrix: columns "<wt> <pos>" (e.g. 'M 1'), rows = mutant amino acid.
    rows = []
    for col in m.columns:
        wt, pos = col.split(" ")
        pos = int(pos)
        for mut in m.index:
            if mut == wt:
                continue
            val = m.at[mut, col]
            if pd.isna(val):
                continue
            rows.append((f"{wt}{pos}{mut}", wt, pos, mut, round(float(val), 4)))
    df = pd.DataFrame(rows, columns=["protein_variant", "wt_aa", "position", "mt_aa", "esm1b_score"])
    df = df.sort_values("position").reset_index(drop=True)
    df["source"] = "REAL"
    df.to_csv(ESM1B_TSV, index=False)
    ESM1B_RELEASE_JSON.write_text(json.dumps({
        "zip_member": ESM1B_MEMBER,
        "zip_member_packaged_at": packaged_at,
        "note": "packaged_at is the zip's own embedded per-file timestamp (zipfile.ZipInfo.date_time), "
                "not a download date -- it is ntranoslab's own record of when this exact CSV was built.",
    }, indent=2))
    print(f"REAL ESM1b CFTR variants written: {len(df):,} -> {ESM1B_TSV.relative_to(DATA_DIR.parent)}")
    print(f"wrote {ESM1B_RELEASE_JSON.relative_to(DATA_DIR.parent)}")


P13569_LLR.csv packaged at (zip-embedded timestamp): 2022-05-06T05:28:04


REAL ESM1b CFTR variants written: 28,120 -> data\esm1b_cftr.csv
wrote data\esm1b_cftr.release.json


In [3]:
esm = tk.load_esm1b()      # REAL — full CFTR saturation LLR (~28,120), built by the cell above
print(f"{len(esm):,} REAL ESM1b variants | source: {esm['source'].unique().tolist()}")
print('LLR range:', esm['esm1b_score'].min(), '->', esm['esm1b_score'].max(),
      '| pathogenic (<= -7.5):', int((esm['esm1b_score'] <= -7.5).sum()))
print(f"esm1b_release (zip-embedded timestamp): {esm['esm1b_release'].iloc[0]}")
esm.head(8)


28,120 REAL ESM1b variants | source: ['REAL']
LLR range: -23.793 -> 4.943 | pathogenic (<= -7.5): 14948
esm1b_release (zip-embedded timestamp): 2022-05-06T05:28:04


,protein_variant,esm1b_score,esm1b_release,source
0,M1K,-5.762,2022-05-06T05:28:04,REAL
1,M1R,-6.544,2022-05-06T05:28:04,REAL
2,M1H,-8.114,2022-05-06T05:28:04,REAL
3,M1E,-6.197,2022-05-06T05:28:04,REAL
4,M1D,-6.949,2022-05-06T05:28:04,REAL
5,M1N,-6.897,2022-05-06T05:28:04,REAL
6,M1Q,-7.112,2022-05-06T05:28:04,REAL
7,M1T,-6.903,2022-05-06T05:28:04,REAL


## 2 · Turn an ESM1b LLR into a call

`tk.call_from_score(score, 'esm1b')` bakes in the **direction**: cut at `<= -7.5`, and **lower / more negative = worse**. You do not juggle the sign yourself.

In [4]:
esm['esm1b_call'] = esm['esm1b_score'].apply(lambda s: tk.call_from_score(s, 'esm1b'))
esm[['protein_variant', 'esm1b_score', 'esm1b_call', 'source']].head(12)

,protein_variant,esm1b_score,esm1b_call,source
0,M1K,-5.762,benign,REAL
1,M1R,-6.544,benign,REAL
2,M1H,-8.114,pathogenic,REAL
3,M1E,-6.197,benign,REAL
4,M1D,-6.949,benign,REAL
5,M1N,-6.897,benign,REAL
6,M1Q,-7.112,benign,REAL
7,M1T,-6.903,benign,REAL
8,M1S,-6.292,benign,REAL
9,M1C,-8.059,pathogenic,REAL


### The sign flip, made concrete

It is worth staring at *one* variant until the backwards scale clicks. Run the next cell.

In [5]:
# Most negative ESM1b = most damaging.
row = esm.sort_values('esm1b_score').iloc[0]
print(f"Variant {row['protein_variant']}:")
print(f"  ESM1b = {row['esm1b_score']:>6}  ->  {row['esm1b_call']:<10} "
      f"(LOW / negative score, cut at -7.5, so low = pathogenic)")
print('A more negative LLR = a more surprising mutation = more damaging —')
print('the opposite direction to EVE, AlphaMissense and REVEL.')

Variant R289P:
  ESM1b = -23.793  ->  pathogenic (LOW / negative score, cut at -7.5, so low = pathogenic)
A more negative LLR = a more surprising mutation = more damaging —
the opposite direction to EVE, AlphaMissense and REVEL.


## Example: the shared missense worked-example panel, scored by **ESM1b**

The same fixed panel of famous CFTR **missense** variants runs through every missense tool
(tools/01–06, benchmark/00–01), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how ESM1b is joined onto it.

In [6]:
panel = tk.A1_PANEL_VARIANTS
esm = tk.load_esm1b()
esm[esm['protein_variant'].isin(panel)][['protein_variant', 'esm1b_score']].reset_index(drop=True)

,protein_variant,esm1b_score
0,G85E,-11.889
1,R117H,-5.659
2,Y161C,-8.857
3,P205S,-7.009
4,R334W,-8.179
5,V520F,-18.170
6,G551D,-12.123
7,R668C,-9.968
8,S912L,-3.418
9,H949Y,-8.836


## Key takeaways

1. **ESM1b** scores the mutant-vs-wild-type **log-likelihood ratio**; cut `<= -7.5` ~ pathogenic — **lower / more negative = worse** (opposite to EVE). `tk.call_from_score` handles the sign.
2. **Unsupervised** → low circularity vs ClinVar (like EVE).
3. This notebook now uses **REAL ESM1b** — full CFTR saturation (~28,120 variants, P13569), `source == REAL`.
4. **Version-tracked despite the manual download**: the build cell reads the release zip's own embedded per-file timestamp (`esm1b_release` = 2022-05-06T05:28:04 for P13569_LLR.csv) rather than leaving the release undated.

